# 05 · Knowledge graphs and evidence paths

**Prerequisites:** Nested loops, sets, and dictionaries. No graph database is required.

**Learning objectives:** Traverse typed relationships; preserve multi-hop provenance; handle ambiguity and missing evidence.

**Guide companion:** sections 9 in `LANGCHAIN_LANGGRAPH_LEARNING_GUIDE.md` at the project root.

**How to work:** Run setup, implement each challenge, then run its acceptance cell. Starter functions deliberately raise `NotImplementedError`; this is expected until you complete them. Restart the kernel and run all cells after finishing. You do not need any other notebook or paid API calls. Budget about 45–90 minutes, or longer for the capstone.

Complete solutions are kept in the matching notebook under `solutions/`. There are no hidden solution cells in this notebook.


In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

# Fictional test data, not real people, policies, or research sources.
NOTES = [
    {"id": "s1", "url": "fixture://architecture", "text": "Cedar uses LangGraph to route research tasks."},
    {"id": "s2", "url": "fixture://review", "text": "Cedar pauses its workflow for human review."},
    {"id": "s3", "url": "fixture://ownership", "text": "Mira maintains Cedar."},
    {"id": "s4", "url": "fixture://team", "text": "Mira works on team Atlas."},
    {"id": "s5", "url": "fixture://policy", "text": "Atlas reviews Cedar evidence monthly."},
]
BY_ID = {note["id"]: note for note in NOTES}

EDGES = [
    ("Cedar", "maintained_by", "Mira", "s3"),
    ("Mira", "member_of", "Atlas", "s4"),
    ("Atlas", "reviews", "Cedar", "s5"),
]
# An edge is (subject, relationship, object, source_id).


## Challenge 1 · Traverse a two-hop relationship

Implement `maintainer_teams(project, edges)`. Follow only `maintained_by` then `member_of`, in those directions. Return every unique path as a dictionary with `project`, `maintainer`, `team`, and `source_ids` (the two source IDs in traversal order).
Deduplicate identical paths and sort by maintainer, team, then source IDs. Unknown projects return `[]`. Do not follow the `reviews` edge or assume only one maintainer exists.


In [ ]:
def maintainer_teams(project: str, edges: list[tuple]) -> list[dict]:
    raise NotImplementedError("Challenge 1: find all typed two-hop paths")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
paths = maintainer_teams("Cedar", EDGES)
assert paths == [{"project": "Cedar", "maintainer": "Mira", "team": "Atlas", "source_ids": ["s3", "s4"]}]
assert maintainer_teams("Unknown", EDGES) == []
assert maintainer_teams("Atlas", EDGES) == [], "Do not confuse reviews with maintained_by"
extended = EDGES + [
    ("Cedar", "maintained_by", "Noor", "s6"),
    ("Noor", "member_of", "Birch", "s7"),
    ("Mira", "member_of", "Atlas", "s4"),
]
many = maintainer_teams("Cedar", extended)
assert [(p["maintainer"], p["team"]) for p in many] == [("Mira", "Atlas"), ("Noor", "Birch")]
assert many[1]["source_ids"] == ["s6", "s7"]
assert maintainer_teams("Cedar", list(reversed(extended))) == many, "Ordering must not depend on input order"
print("PASS: typed traversal, all matching paths, deduplication, and stable ordering")


## Challenge 2 · Recover the supporting documents

Implement `collect_evidence(paths, notes)`. Return copies of all referenced notes, deduplicated and sorted by source ID. If any path references an unavailable source, raise `ValueError` and identify the missing ID in its message. Do not silently drop unsupported edges.


In [ ]:
def collect_evidence(paths: list[dict], notes: list[dict]) -> list[dict]:
    raise NotImplementedError("Challenge 2: resolve every path citation to evidence")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
paths = maintainer_teams("Cedar", EDGES)
evidence = collect_evidence(paths + paths, NOTES)
assert [n["id"] for n in evidence] == ["s3", "s4"]
assert collect_evidence([], NOTES) == []
evidence[0]["text"] = "changed"
assert BY_ID["s3"]["text"] == "Mira maintains Cedar."
try:
    collect_evidence(paths, [BY_ID["s3"]])
except ValueError as exc:
    assert "s4" in str(exc)
else:
    raise AssertionError("Missing path provenance must fail visibly")
print("PASS: complete, copied, deduplicated provenance")


## Challenge 3 · Extend the graph without losing provenance

Implement `add_second_maintainer(notes, edges)` returning `(new_notes, new_edges)` without modifying its inputs. Add exactly these fictional facts:

- `s6`, `fixture://ownership-noor`: `Noor maintains Cedar.`
- `s7`, `fixture://team-noor`: `Noor works on team Birch.`

Add the corresponding `maintained_by` and `member_of` edges. Preserve the existing graph and collection. You may assume these IDs and edges are not already present.


In [ ]:
def add_second_maintainer(notes: list[dict], edges: list[tuple]) -> tuple:
    raise NotImplementedError("Challenge 3: extend the graph and its source collection together")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
new_notes, new_edges = add_second_maintainer(NOTES, EDGES)
assert len(NOTES) == 5 and len(EDGES) == 3, "Do not mutate fixtures"
assert len(new_notes) == 7 and len(new_edges) == 5
expanded_paths = maintainer_teams("Cedar", new_edges)
assert {p["team"] for p in expanded_paths} == {"Atlas", "Birch"}
assert [n["id"] for n in collect_evidence(expanded_paths, new_notes)] == ["s3", "s4", "s6", "s7"]
print("PASS: two maintainers, two teams, four supporting notes")


## Reflection

1. How does this data graph differ from the execution graph in LangGraph?
2. Does this notebook implement Microsoft GraphRAG? What is missing?
3. How would you distinguish two people named Mira or historical ownership from current ownership?


**Your answers:**

Write your reasoning here before opening the solutions.


## References

- [Graph database concepts](https://neo4j.com/docs/getting-started/appendix/graphdb-concepts/)
- [Microsoft GraphRAG indexing](https://microsoft.github.io/graphrag/index/overview/)
- [GraphRAG local search](https://microsoft.github.io/graphrag/query/local_search/)
